FILE 1: `policy.py` -- THE CONTRACT

LINES 90-134 -- `set_pose_target()` -- THE CONVENIENCE HELPER

```Python
    def set_pose_target(
        self, move_robot: MoveRobotCallback, pose: Pose, frame_id: str = "base_link"
    ) -> None: 
```
   This wraps up a `MotionUpdate` message with sensible defaults. The key 
   parameters it sets:

```Python
    motion_update = MotionUpdate(
        target_stiffness=...,
        target_damping=...,
    )
```
   * STIFFNESS ...: How strongly the arm tracks the target pose. First 3 = 
     translational (N/m), last 3 - rotational (Nm/rad). Higher = stiffer 
     tracking, more jerk.
   * DAMPING ...: Velocity-proportional resistance. Higher = smoother but slower

```Python
    wrench_feedback_gains_at_tip = ... 
```
    Force feedback gains. [0.5, 0.5, 0.5] means 50% force feedback on 
    translation (the arm yields to external forces). [0, 0, 0] on rotational
    means no rotationalc ompliance. Range is [0, 0.95].

```Python
    trajectory_generation_mode=TrajectoryGenerationMode(
        mode=TrajectoryGenerationMode.MODE_POSITION,
    ),
```
   `MODE_POSITION` means the pose field is used. `MODE_VELOCITY` means the
   velocity field is used instead.


LINES 136-145 -- THE ABSTRACT METHOD YOU IMPLEMENT
```Python
    @abstractmethod
    def insert_cable(
        self,
        task: Task,                                 # What to insert where
        get_observation: GetObservationCallback,    # See the world
        move_robot: MoveRobotCallback,              # Command the arm
        send_feedback: SendFeedbackCallback,        # Report progress
    ) -> bool:                                      # True = success
```



---
FILE2: `aic_model.py` -- THE LIFECYCLE NODE (Your Runtime)
   PATH: `aic_model/aic_model/aic_model.py` (335 lines)
   IMPORTANCE: Understand this to know waht happens around your code.

KEY LINES EXPLAINED

Lines 53-80 -- Dynamic Policy Loading

```Python
class AicModel(LifecycleNode):
    def __init__(self):
        super().__init__("aic_model")
        self.declare_parameter("policy", "WaveArm")
        policy_module_name = (
            self.get_parameter("policy").get_parameter_value().string_value
        )
        policy_module = importLib.import_module(policy_module_name)
```
   Your policy is loaded dynamically via string name. When you run 
   `ros2 run aic_model aic_model --ros-args -p policy:=my_package.MyPolicy`,
   it imports `my_package.MyPolicy` and looks for a class named `MyPolicy` 
   inside.

   
LINES 81-84 -- TF BUFFER SETUP

```Python
    self._tf_buffer = Buffer()
    self._tf_listener = TransformListener(
        buffer=self._tf_buffer, node=self, spin_thread=True
    )
```
   The TF buffer accumulates transform data from `/tf` and `/tf_static`. Your
   policy accesses this via `self._parent_node._tf_buffer`. When 
   `ground_truth:=true`, cable and port TF frames are published here.


LINES 91-93 -- OBSERVATION SUBSCRIPTION
```Python
    self.observation_sub = self.create_subscription(
        Observation, "observations", self.observation_callback, 10
    )
```
   Subscribes to the `/observations` topic (published by `aic_adapter` at 20Hz).
   When your policy calls `get_observation()`, it gets whatever was last stored.


LINES 190-229 -- AUTOMATIC MODE SWITCHING
```Python
    def handle_motion_update(self, motion_update: MotionUpdate):
        if self._target_mode != TargetMode.MODE_CARTESIAN:      # Line 191
            self.set_target_mode(TargetMode.MODE_CARTESIAN)     # Line 193
        self.motion_update_pub.publish(motion_update)    
```
   If you send a `MotionUpdate` but your controller is in joint mode, it 
   automatically calls the `ChangeTargetMode` service to switch.


LINES 236-244 -- HOW YOUR POLICY GETS CALLED
   Your `insert_cable()` runs in a separate thread (`action_thread_func`). The
   main thread continues spinning ROS callbacks. This is why `get_observation()`
   always returns the latest data even while your policy is in a loop.


---
FILE 3: `Observation.msg` -- EVERYTHING YOU CAN SEE
   PATH: `aic_interfaces/aic_model_interfaces/msg/Observation.msg`
   IMPORTANCE: This is your sensor input -- every perception decision depends on
   this.

   * ... (* wrist camera RGB 480x640x3)
   * `geometry_msgs/WremchStamped_wrist_wrench` (6-axis F/T sensor)
   * `sensor_msgs/JointState joint_states` (7 values: 6 arm joints + 1 gripper)
   * `aic_control_interfaces/ControllerState controller_state` ()

---
FILE 4: `ControllerState.msg` -- The Robot's Self-Knowledge
   PATH: `aic_interfaces/aic_control_interfaces/msg`
   IMPORTANCE: Gives you computed TCP state without doing FK yourself. 

   * `geometry_msgs/Pose tcp_pose` (Current gripper TCP position + orientation)
   * `geometry_msgs/Twist tcp_velocity` (Current TCP linear + angular velocity)
   * `geometry_msgs/Pose reference_tcp_pose` (Where the controller is TRYING to
     go)
   * `float64[6] tcp_error` ([x, y, z, rx, ry, rz] error from current to target)
   * `trajectory_msgs/JointTrajectoryPoint reference_joint_state`
   * `TargetMode target_mode` (Current control mode)

---
FILE 5: `MotionUpdate.msg` -- How you Move the Arm (Cartesian)
   ...
   IMPORTANCE: This is your primary command interface for precise manipulation.

   * Target position + quaternion orientation
   * Target linear + angular velocity
   * 6x6 impedance stiffness, row-major
   * 6x6 impedance damping, row-major
   * Fprce/torque to apply at TCP
   * Force feedback gains at [0-0.95] 
   * POSITION or VELOCITY mode     

   WHAT YOU'LL MODIFY
      You'll create `MotionUpdate` messages in your policy with carefully tuned
      stiffness/damping for each phase of cable insertion:
      
      * APPROACH PHASE: Medium stiffness, high damping (smooth approach)
      * ALIGNMENT PHASE: Lower stiffness, high wrench gains (force-guided
        alignment)
      * INSERTION PHASE: High z-stiffness, low xy-stiffness, feedforward z-force
        (push in while staying compliant laterally).  


---
FILE 6: `Task.msg` -- What You're Asked To Do
   
   * `string id` - Unique trail identifier
   * `string cable_type` - e.g., "sfp_sc_cable"
   * `string cable_name` - e.g., "cable_0"
   * `string plug_type`  - e.g., "sfp"
   * `string plug_name`  - e.g., "sfp_tip"
   * `string port_type`  - e.g., "sfp"
   * `string port_name`  - e.g., "sfp_port_0"
   * `string target_module_name` - e.g., "nic_card_mount_0"
   * `uint64 time_limit` - Typically 180s

   This tells your policy: "Take cable `cable_name`, specifically its plug end
   `plug_name`, and insert it into port `port_name` on module 
   `target_module_name`".

   The `CheatCode` policy uses these to construct TF frame names:

```Python
port_frame = f"task_board/{task.target_module_name}/{task.port_name}_link"  # Line 197
cable_tip_frame = f"{task.cable_name}/{task.plug_name}_link"                # Line 198
```
   
   WHAT YOU'LL MODIFY
      You won't modify the message. But your policy must handle ALL COMBINATIONS:
         * SFP into NIC card 
         * SFP into SFP ports
         * SC into SC ports
         * SC into reversed cables,...
      The task board layout changes between trails, so you can't hardcode 
      positions.

---

FILE 7: `CheatCode.py` - THE GOLD STANDARD STRATEGY
   PATH: `aic_example_policies/aic_example_policies/ros/CheatCode.py`
   IMPORTANCE: This shwos the OPTIMAL motion strategy. Study this like a 
   playbook.

   KEY METHODS EXPLAINED

   Line 45-70 -- `_wait_for_tf()` -- Blocking TF Lookup




   THE QUATERNION LOGIC: To align the plug with the port, compute the rotation
   difference between them. `q_diff = q_port * q_plug_inverse`. Then apply this
   rotation to the gripper:

   ```Python
   q_port = (port_transform.rotation.w, .x, .y, .z)      # Line 81-86

   ...
   ```



   ...


   ```Python
   q_gripper_target = quaternion_multiply(q_diff, q_gripper)
   q_gripper_slerp = quaternion_slerp(q_gripper, q_gripper_target, slerp_fraction)
   ```

   SLERP (Spherical Linear Interpolation) smoothly rotates from current to 
   target orientation. `slerp_fraction` goes from 0.0 (current) to 1.0 (aligned)


   ...

   The plug tip and gripper TCP are NOT the same point. This offset accounts for
   the cable hanging from the gripper - the plug is below/beside the TCP.


   ...

   PI CONTROLLER for xy alignment. The plug doesn't perfectly follow the gripper
   (cable flexibility), so a proportional-integral loop corrects the drift. The
   integrator is clipped at +-0.05m to prevent windup.


   ...

LINES 187-258 - `insert_cable()`

   Over 100 steps (5 seconds), smoothly interpolate from wherever the arm is to 
   a position 20cm DIRECTLY ABOVE THE PORT, while simultaneously aligning the
   plug orientation to match the port.

---

   
   Grasping how TF (which stands for TRANSFORM) works is practically a rite of
   passage in ROS 2.

   At its core, TF IS THE ROBOT'S SPATIAL AWARENESS SYSTEM. It is a library that
   keeps track of where every single part of the robot (and the world) is
   relative to every other part, at any given millisecond.

   ... why it matters for your cable insertion task.


---
THE PROBLEM TF SOLVES
   Imagine your UR5e robot is trying to plug an SFP cable into a NIC card.
   * The CAMERA sees the NIC card port. It says: "The port is 10 cm directly in
     front of my lens."
   * But the GRIPPER needs to mvoe to that port.
   * The robot's BASE is bolted to the table and is what actually commands the
     motors.

   How does the robot's base know how to move the gripper to a spot that was 
   seen by the camera? It needs a massive chain of 3D math (translations and
   rotations) connecting the Camera to the Wrist, the Wrist to the Elbow, the
   Elbow to the Shoulder, and the Shoulder to the Base.

   TF handles all that math automatically in the background. 



---
HOW TF WORKS: The Tree
   TF organises all these different coordinate systems (called FRAMES) into a
   giant parent-child tree.

   In the AIC simulation, your TF Tree looks something like this:

   * `world` (The absolute center of the simulation)
      * `task_board`
         - `nic_card_0`
            - `sfp_port_0_link` (The Target)
      * `base_link` (The bottom of your UR5e)
         - `shoulder_link`
            _ `wrist_3_link`
               * `gripper/tcp` (The Tool Center Point)
               * `left_camera_optical_frame`



THE `tf_buffer` AND `tf_listener`
   In `aic_model.py`, you saw these two lines:

```Python
self._tf_buffer = Buffer()
self._tf_listener = TransformListener(buffer=self._tf_buffer, node=self)
```
   
   - THE LISTENER: The robot is constantly broadcasting the angles of its
     joints. The `TransformListener` listens to these broadcasts (on the `/tf`
     topic) in real-time.
   - THE BUFFER: It stores a short history (usually a few seconds) of all these
     frames. This is crucial because if you want to know where the camera was
     exactly when an image was taken 50 milliseconds ago, you ask the buffer.



HOW YOU USE IT (The `lookup_transform` command)
   This is where the magic happens in the `CheatCode.py` policy.

   Instead of doing complex matrix multiplication yourself, you just ask the TF
   Buffer: "Hey, what is the exact distance and rotation from the robot's base
   to the SFP port right now?"

```Python
port_tf = self._tf_buffer.lookup_transform(
   target_frame="base_link",           # I want the answer relative to this frame
   source_frame="sfp_port_0_link",     # I want to know where this frame is
   time=Time()                         # Get the most recent data
)
```

   The TF system climbs up and down the tree, multiplies all the translation
   vectors and rotation quaternions together, and hands you back a single, clean
   `Transform` object. You then use that transform to tell your gripper exactly
   where to go via a `MotionUpdate`.

   This is a great technical question. The graph you provided is an 
   architectural overview of ROS NODES and TOPICS, but it doesn't show the
   INTERNAL DATA STRUCTURES living inside those nodes,

   The `tf_buffer` and `tf_listener` are not external components; they live
   INSIDE YOUR `aic_model` NODE (the blue box labeled "YOUR SUBMISSION").

   Here is the breakdown of where they sit and why the graph might feel 
   "incomplete":


1. WHERE THEY SIT IN THE GRAPH
   Look at the circles on the left and right labeled `/tf` and `/tf_static`.
   * `tf_listener`: This is an internal "ear" inside your `aic_model` node. It
     creates a background thread that constantly "subscribes" to those `/tf`
     and `/tf_static` circles.
   * `tf_buffer`: This is an internal "memory" (a cache) inside your `aic_model`
     node. The listener takes every message it hears from the topics and stores
     them in this buffer so you can query them later.


2. IS THE GRAPH INCOMPLETE?
   The graph is COMPLETE FROM A SYSTEM COMMUNICATION PERSPECTIVE, but it 
   abstracts away the internal logic. To visualise where TF goes, imagine a
   dotted line:
   1. From `/tf` and `/tf_static` (The blue circles)
   2. To `aic_model` (Your blue submission box)

   Inside that box, your `tf_listener` is what bridges those external topics 
   into your local `tf_buffer`.


3. WHY THIS MATTERS FOR THE COMPETITION
   If you look closely at the circle for `/tf`, there is a red warning that
   says: "FORBIDDEN in eval!"
   * IN TRAINING: You can use the `tf_buffer` to look up the exact "Ground 
     Truth" position of the SFP port relative to the world.
   * IN EVALUATION: The organisers will stop publishing the environment frames
     to the `/tf` topic. Your `tf_listener` will still hear the robot's own
     joints (so you know where your gripper is), but the "memory" in your
     `tf_buffer` for the SFP port will be empty.

This is why the competition requires you to use the CAMERAS (the orange lines
from `aic_adapter`) to "see" the port, rather than just relying on the TF buffer
to "know" where it is.     
          







   The reason the organisers stop publishing environment frames like the SFP
   PORT or NIC CARD locations is to prevent "cheating" via ground truth data. In
   a simulation, the engine knows exactly where every object is down to the
   milimeter, and it can broadcast those coordinates over the `/tf` topic. If
   those frames were available durin evaluation, you wouldn't need AI or
   computer vision at all; you would simply write a script to move the gripper 
   to the exact mathematical coordinate of the port. This would defeat the
   entire purpose of the challenge, which is to develop a policy that can
   perceive and manipulate objects using only the robot's onboard sensors--the
   three wrist cameras and the force/torque sensors.

   During evaluation, the ZENOH ACCESS CONTROL system acts as a filter. It 
   allows the `/tf` and `/tf_static` messages describing the robot's own body 
   parts to reach your `aic_model` because the robot needs to know where its
   "hand" is relative to its "shoulder" to move correctly. However, it "blocks"
   any messages that describe the task board components. Consequently, while
   your `tf_listener` continues to populate the `tf_buffer` with the robot's
   internal kinematics, any attempt to look up the transform for a port will
   return an error because that specific "branch" of the spatial tree has
   been physically cut off from the stream.

   This forced transition from GROUND TRUTH to SENSOR-BASED PERCEPTION is what
   bridges the "sim-to-real" gap. In the real world, a server rack doesn't 
   broadcast its exact GPS coordinates to a robotl the robot msut look at the
   rack, identify the SFP cage using its vision model, and then calculate the
   necessary path. By disabling the environment's TF frames, the competition
   ensures that your solution is actually "seeing" the world through the orange
   camera lines in the graph, making is robust enough to handle the randomised
   positional offsets and translations specified in the trail configurations.




















---

-- In ROS2, FK stands for Forward Kinematics. It is a fundamnetal robotics
   concept used to calculate the position and orientation (the pose) of a 
   robot's end-effector (e.g., a gripper or hand) based on known joint angles or
   displacements.

   In essence, FK takes joint states as input and provides the Cartesian 
   position of the robot's links in space as output.

---

    * THE SUPREME ENTRY POINT: ./aic/aic_model/aic_model/aic_model.py
            - This contains the main function, and it has a series of scalpers
              from lines 62-122 which extracts the various input parameters
              given from the entry command.
`ros2 run aic_model aic_model --ros-args -p policy:=aic_example_policies.ros.CheatCode`
            - aic_model.py's AicModel(LifecycleNode) class then scalps the class
              definition of `aic_example_policies.ros.CheatCode` in line 67
              `inspect.getmembers(policy_module, inspect.isclass)`.
            - The `CheatCode.py`'s ckass is defined as CheatCode(policy) --
              which implements from the interface in 
              `./aic/aic_model/aic_model/policy.py`... and hence the full circle
              is reached and everything ties together nicely.
          
            - Which all the client-side (our side)'s configs are passed in from
              `class CheatCode(policy)` ... which this is where we implement the
              list of 10 policy models, and other fun stuff too...
              

the main entry point seems to be `aic_model.py` the policy interface file is#
`policy.py`

our custom policy can be implemented anywhere in the directory (though our 
custom policy class itself implements from the interface class `Policy` from
`Policy.py`)


---

    But in the ROS2 command that we shoved into terminal: 
`ros2 run aic_model aic_model --ros-args -p policy:=aic_example_policies.ros.CheatCode`

    This tells `aic_model.py` to treat `aic_examnple_policies/ros/CheatCode.py`
    as the policy .... 
    
    ...


---

    so yeah, the rest are just `.msg` files which are intermediary kinda 
    serialised objects pass between functions or stored during operation

    or template examples files for ACT (action chunking with transformers <-- I
    wonder can we change to VLA from here or sth... but likely not useful 
    because we have a dead fixed task here...), or the cheat code file... which
    from what im seeing basically are just: `RunACT.py` 









---

PART 1: WHAT EXACTLY IS AN "ACT"?
    ACT stands for ACTION CHUNKING WITH TRANSFORMERS. It is currently one of the
    most popular and powerful Imitation Learning (IL) architectures for 
    real-world robotic manipulation, originally developed...

    To understand why it's a breakthrough, you have to understand the problem
    it solves:
    
   1. THE COMPOUNDING ERROR PROBLEM: Older AI models predicted robot movement
      one single step at a time (e.g., "move 1mm left"). If the robot made a 
      tiny mistake, the next camera frame would look slightly weird, causing it
      to make a bigger mistake on the next step. The robot would quickly jitter
      and fail.
   2. THE "CHUNKING" SOLUTION: Instead of predicting 1 step, ACT predicts a 
      "CHUNK" of future actions--usually the next 50 to 100 steps all at once.
   3. TEMPOLAR ENSEMBLING: At step 1, it predicts steps 1-100. At step 2, it
      predicts steps 2-101. It overlaps these predictions and averages them out.
      This makes the robot's movements incredibly smooth and dramatically 
      reduces jitter and errors.
   4. THE TRANSFORMER: It uses a Transformer (similar architecture to LLMs) to
      process multiple camera viewpoints (left, center, right) and the robot's
      join states simultaneously to understand the 3D spatial relationship of
      the scene.

   IN SHORT: You teleoperate the robot to insert the cable 50 times. You feed 
   that data into ACT. ACT learns to mimic your exact fluid, multi-step motions
   using camera feeds.


---
PART 2: DECONSTRUCTING `RunACT.py`
    This file is a bridge. It takes the AI model you trained using Hugging 
    Face's `lerobot` library and plugs it into the Intrinsic AI Challenge
    simulator.

    Here is what the code is doing, step-by-step:

   1. THE SETUP && LOADING (Lines 55-90)
       * WHAT IT DOES: It connects to the HF internet hub, downloads a specific
         pre-trained ACT model (`grkw/aic_act_policy`), and loads the neural
         network weights into your GPU (`self.device = torch.device("cuda")`).
       * WHY IT MATTERS: This means you don't have to package gigabytes of model
         weights into your Docker container. It pulls your brain from the cloud
         at runtime. 
   2. THE GREAT NORMALISATION (Lines 91-168)
       * WHAT IT DOES: AI models don't understand raw pixels (0-255) or raw 
         robot coordinates. They need numbers squashed between roughly -1 and 1.
         This section loads the MEAN and STANDARD DEVIATION (std) of the data
         you used during training, and applies the formula `(x - mean) / std` to
         the live camera and robot data. 
       * WHY IT MATTERS: If you skip this, or use the wrong stats, your robot 
         will flail wildly becuse the input numbers look completely alien to the
         neural network.
   3. DATA FORMATTING (Lines 169-236)
      * WHAT IT DOES: The `prepare_observations` function takes the nice ROS
        `Observation.msg` and shreds it into raw NumPy/PyTorch arrays. It stacks
        the TCP position, orientation, velocities, and joint states into one
        massive 26-dimensional flat array.             
   4. THE INFERENCE LOOP (Lines 237-296)
      * WHAT IT DOES: This is the `insert_cable` function that runs the trail.
         * It grabs the cameras/state (`get_observation()`).
         * Prepares the data.
         * LINE 266 IS THE MAGIC: 
            `normalized_action = self.policy.select_action(obs_tensor)`
           The AI looks at the cameras and spits out what to do next.
         * It un-normalises the output back into real-world speeds.
         * It extracts a 6D velocity vector (Linear X/Y/Z + Angular X/Y/Z)
         * It loops at 4Hz (`time.sleep(0.25)`).
   5. THE COMMAND EXECUTION (Lines 297-321)
      * WHAT IT DOES: It packages that 6D velocity into a `MotionUpdate` message
      * CRUCIAL DETAIL: Notice on L317 it sets 
        `mode=TrajectoryGenerationMode.MODE_VELOCITY`. Unlike the CheatCode
        policy which commands positions, ACT commands the robot's speed and
        direction, riding it like a joystick.


---
PART 3: WHEN, HOW, AND WHY TO USE THIS FILE

WHEN WILL YOU USE IT?
   You will use this file IF YOU DECIDE TO USE IMITATION LEARNING for the 
   competition of your hackathon.

   If your plan is: "I'm going to set up my Xbox controller or VR headset,
   teleoperate the simulated robot to plug in the SFP cable 100 times, and have
   an AI copy me" -- this is exactly the file you will use to run that AI.

HOW WILL YOU MODIFY IT?

   1. LINE 62 (`repo_id = "grkw/aic_act_policy"`): You will change this to point
      to your HF account (e.g., ...) ...

   2. LINE 198-228 (`state_np = np.array(...)`): If you decide you only want
      your AI to look at TCP positions and ignore joint states to make training
      faster, you will change this array to match whatever you trained on.

   3. LOGIC ROUTING: As we discussed earlier, you might add an `if` statement in
      `insert_cable` to load a different `repo_id` depending on if 
      `task.plug_type` is SFP or SC.

   WHY US IT OVER OTHER METHOD?      
      * PROS OF ACT: It is phenomenally good at contact-rich tasks (like 
        sldiding a plug into a tight port). It handles the "wiggle" natually if 
        your human demonstrations include wiggling. It doesn't require compleax 
        math or coding the exact physics of the port.
      * CONS OF ACT: You have to collect the data manually. Gathering 100
        successful human demonstrations of cable insertion in simulation can
        take hours of tedious, repetitive work.

   WHY NOT USE IT?
      If you decide to solve this using RL (where the robot learns by trail and
      error over millions of simulation steps) or CLASSICAL CONTROL (using 
      OpenCV to find the port and writing math to calculate the perfect 
      trajectory, like `CheatCode.py` does), you will ignore `RunACT.py` 
      entirely.




                    


